In [ ]:
# ============================================
# DEEPFAKE DETECTION - 3-DATASET T=16 TRAINING
# Experiment A5: A5 (Concat Fusion)
# ============================================
# Datasets: DFDC02 + DFD01 + CelebDF (all preprocessed with T=16)
# GPU: P100 recommended (16GB)
#
# Required Kaggle inputs:
#   - dfdc02-preprocessed (T=16)
#   - dfd01-preprocessed (T=16)
#   - celebdf-preprocessed (T=16)
#
# Estimated GPU time: ~3 hours

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc, shutil
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
import torch

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'

# Find all T=16 datasets
TARGET_T = 16
print(f'=== Searching for T={TARGET_T} datasets ===')
found_datasets = []
for root, dirs, files in os.walk('/kaggle/input'):
    dirname_lower = os.path.basename(root).lower()
    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            sample_dir = None
            for label in ('real', 'fake'):
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                n_frames = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg')])
                if n_frames > 0 and abs(n_frames - TARGET_T) > TARGET_T * 0.5:
                    continue

            path_lower = root.lower()
            if 'dfd01' in path_lower or 'dfd-01' in path_lower or 'dfd_01' in path_lower:
                ds_name = 'dfd01'
            elif 'dfdc02' in path_lower or 'dfdc-02' in path_lower or 'dfdc' in path_lower:
                ds_name = 'dfdc02'
            elif 'celeb' in path_lower:
                ds_name = 'celebdf'
            else:
                continue

            if any(d['name'] == ds_name for d in found_datasets):
                continue

            found_datasets.append({'path': root, 'name': ds_name, 'real': rc, 'fake': fc})
            print(f'  Found: {ds_name} at {root} (real={rc}, fake={fc})')

assert len(found_datasets) >= 3, f'Need 3 datasets, found {len(found_datasets)}'

dataset_roots = '+'.join(d['path'] for d in found_datasets)
dataset_names = '+'.join(d['name'] for d in found_datasets)
print(f'Combined: {dataset_names}')

from config import Config
from train import train
from utils import save_metrics

EXP_NAME = 'A5_concat_fusion'

print(f'[A5] Starting training...')

cfg = Config()
cfg.dataset_root = dataset_roots
cfg.dataset_name = dataset_names
cfg.model_type = 'full'
cfg.fusion_type = 'concat'  # <<< KEY: this is A5 ablation
cfg.num_frames = 16
cfg.min_frames_per_video = 12
cfg.max_epochs = 30
cfg.batch_size = 16
cfg.patience = 7
cfg.device = 'auto'
cfg.use_amp = True
cfg.num_workers = 2
cfg.pin_memory = True
cfg.splits_dir = './splits'
cfg.output_dir = './experiments'
cfg.seed = 42
cfg.unfreeze_last_n_blocks = 2

t0 = time.time()
try:
    metrics = train(cfg)
    metrics['experiment_name'] = EXP_NAME
    metrics['status'] = 'success'
    metrics['duration_min'] = round((time.time() - t0) / 60, 1)
    test = metrics.get('test', {})
    print(f'[OK] {EXP_NAME} done in {metrics["duration_min"]} min')
    print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
    save_metrics([metrics], f'./experiments/all_results_A5_concat.json')
except Exception as e:
    print(f'[FAIL] {e}')
    import traceback; traceback.print_exc()
    sys.exit(1)

# Save results
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_A5_concat.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'results_A5_concat.tar.gz ready for download')
'''

with open('/kaggle/working/run_training_A5.py', 'w') as f:
    f.write(train_script)

# Step 4: Run
print(f'=== Starting A5 (concat fusion) Training ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_A5.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'Subprocess exited with code: {result.returncode}')
